# PCNN with Piecewise Polynomial Trial Functions (PPTF)

## Author: RIDWAN ADEMOLA IBRAHIM ##

## IDA-KI OpenLab Research Bridge, TU Dresden

### Overview

This notebook uses two separate polynomial trial functions, one for each span, with boundary conditions matched to the corresponding support configuration:

- **Span AB [0,15]:**

  $$
  \phi_{AB}(x)
  =
  \frac{x^2(x-15)^2}{\mathrm{scale}}
  $$

  Fixed-fixed shape with 4 hard boundary conditions.

- **Span BC [15,30]:**

  $$
  \phi_{BC}(\xi)
  =
  \frac{\xi^2(15-\xi)(22.5-\xi)}{\mathrm{scale}}
  $$
  
  where

  $$
  \xi = x - 15.
  $$

  Fixed-pinned shape with 4 hard boundary conditions.

---

### All 8 Boundary Conditions Hard-Coded

| Boundary Condition | Span | Enforced By |
|-------------------|------|-------------|
| w(0) = 0 | AB | Double root at x = 0 |
| w'(0) = 0 | AB | Double root at x = 0 |
| w(15) = 0 | AB | Double root at x = 15 |
| w'(15) = 0 | AB | Double root at x = 15 |
| w(15) = 0 | BC | Double root at ξ = 0 |
| w'(15) = 0 | BC | Double root at ξ = 0 |
| w(30) = 0 | BC | Single root at ξ = 15 |
| w''(30) = 0 | BC | Built into (22.5 − ξ) factor |

---

### Key Differences from SPTF

1. **No Moment Loss**

   The boundary condition

   $$
   w''(30)=0
   $$

   is algebraically enforced by the Span BC trial function, so no separate moment loss term is required.

2. **Continuity Evaluated at x = 13 m and x = 17 m**

   A 2 m offset from the interior support is used because both trial functions are close to zero near x = 15 m.

3. **Three-Component Loss Function**

   - Data Loss
   - PDE Loss
   - Continuity Loss

   Unlike SPTF, no Moment Loss term is required.

4. **Symmetric Span AB Trial Function**

   The Span AB trial function is symmetric about

   $$
   x = 7.5\ \text{m}.
   $$

   Unlike SPTF, there is no \((x-30)\) factor to introduce asymmetry. This symmetry is physically inconsistent with a continuous beam and contributes to degraded performance in Span AB.

---

### Known Limitation

The piecewise construction introduces derivative discontinuities at x = 15 m. The continuity loss penalizes these discontinuities but cannot completely eliminate them.

Compared with SPTF:

- Moment continuity worsens by approximately 8×.
- Shear continuity worsens by approximately 43×.

## 1. Imports, Constants, and Configuration

Core libraries: PyTorch for automatic differentiation (computing w', w'', w''', w'''' through the neural network graph), NumPy/Pandas for data handling, Matplotlib for visualisation.

**Key constants:**
- `EI_NOM = 192,708 kN.m2`: Flexural rigidity (E = 37 GPa, I = 5.208e-3 m4)
- `EI_INV = 1/EI`: Coefficient of q(t) in the normalised beam PDE
- `TILTX_SCALE = 1000`: Converts slope (m/m) to tiltx (mm/m)
- `FOURIER_FREQS = [1,2,4,8,16]`: Dyadic frequencies for temporal/thermal Fourier encoding

**Output directory:** `outputs/processing/Tiltx/Piecewise_Trial_function_PCNN/`

In [7]:
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os, json
from datetime import datetime, timedelta

torch.set_default_dtype(torch.float64)

data_wd = "C:\\Users\\ridoc\\OneDrive\\Desktop\\Folders\\SMACCs\\Thesis\\Physics Guided Framework\\Draft\\cleaned_tilt_data.csv"
FEA_SLOPE_CSV = 'outputs/processing/Tiltx/FEM/reconstructed_slopes.csv'
FEA_DEFL_CSV  = 'outputs/processing/Tiltx/FEM/reconstructed_deflections.csv'
OUT = os.path.join(os.path.dirname(data_wd), "Models","outputs","processing","Tiltx","Piecewise_Trial_function_PCNN")
os.makedirs(OUT, exist_ok=True)

#Same as in SPTF
L_TOTAL = 30.0; L_SPAN = 15.0
EI_NOM = 37.0e6 * 5.20833e-3
EI_INV = 1.0 / EI_NOM
TILTX_SCALE = 1000.0
FOURIER_FREQS = [1, 2, 4, 8, 16]
N_FOURIER = len(FOURIER_FREQS)
VAL_POS = [5.0, 10.0, 20.0, 25.0]
CP = {1: '#2196F3', 2: '#4CAF50', 3: '#FF9800', 4: '#F44336'}
LP = {1: 'Ph1', 2: 'Ph2: +tracks', 3: 'Ph3: +veh4.1t', 4: 'Ph4: +veh9t'}
REF_DATE = pd.Timestamp('2024-02-01').date()
DISTINCT_COLORS = ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4',
                   '#42d4f4', '#f032e6', '#bfef45', '#800000', '#469990']

print(f"  EI = {EI_NOM:.0f} kN.m2")

  EI = 192708 kN.m2


## 2. Fourier Encoding
Same temporal Fourier encoding as SPTF: sin/cos at f=[1,2,4,8,16] for normalised time and standardised temperature.

In [3]:
#Same as SPTF
def fourier_encode(vals, freqs):
    parts = []
    for f in freqs:
        parts.append(torch.sin(2 * np.pi * f * vals))
        parts.append(torch.cos(2 * np.pi * f * vals))
    return torch.cat(parts, dim=1)

## 3. Piecewise Trial Functions

Two separate polynomials, each matching the correct BC type for its span:

**phi_AB(x) = x2(x-15)2 / scale** (fixed-fixed)
- Double root at x=0: w=w'=0
- Double root at x=15: w=w'=0
- Symmetric about x=7.5 (unlike the actual beam shape which is asymmetric due to coupling with span BC)

**phi_BC(xi) = xi2(15-xi)(22.5-xi) / scale** (fixed-pinned), where xi = x-15
- Double root at xi=0 (x=15): w=w'=0
- Single root at xi=15 (x=30): w=0
- The (22.5-xi) factor ensures w''(30)=0 algebraically: the second derivative of xi2(15-xi)(22.5-xi) vanishes at xi=15

Both are normalised by their maximum absolute value for numerical stability.


In [4]:
#Same as SPTF but each span has its normalizing factor
_x = np.linspace(0, L_SPAN, 5000)
PHI_AB_S = float(np.max(_x**2 * (_x - 15)**2))
PHI_BC_S = float(np.max(np.abs(_x**2 * (15 - _x) * (22.5 - _x))))

#Span AB Trial Function
def phi_AB(x):
    return x**2 * (x - 15.0)**2 / PHI_AB_S

#Span BC Trial Function
def phi_BC(x):
    xi = x - 15.0
    return xi**2 * (15.0 - xi) * (22.5 - xi) / PHI_BC_S

print(f"  PHI_AB_S = {PHI_AB_S:.2f}")
print(f"  PHI_BC_S = {PHI_BC_S:.2f}")
for xv in [0, 7.5, 15, 22.5, 30]:
    va = phi_AB(torch.tensor([[float(xv)]])).item() if xv <= 15 else 0
    vb = phi_BC(torch.tensor([[float(xv)]])).item() if xv >= 15 else 0
    print(f"  x={xv:.1f}: phi_AB={va:+.4f}  phi_BC={vb:+.4f}")

  PHI_AB_S = 3164.06
  PHI_BC_S = 6580.59
  x=0.0: phi_AB=+0.0000  phi_BC=+0.0000
  x=7.5: phi_AB=+1.0000  phi_BC=+0.0000
  x=15.0: phi_AB=+0.0000  phi_BC=+0.0000
  x=22.5: phi_AB=+0.0000  phi_BC=+0.9616
  x=30.0: phi_AB=+0.0000  phi_BC=+0.0000


## 4. Data Loading and Preprocessing

Load the cleaned tilt data CSV and prepare the dataset for training.

### Processing Steps

1. **Daily Averaging**

   Sub-daily measurements are aggregated into daily mean values for:

   - Tilt-X
   - Temperature

2. **Phase Assignment**

   Based on known construction and loading stages:

   - **Phase 1 (Day 21–45):** Dead load only (25 days)
   - **Phase 2 (Day 46–80):** Railway tracks installed (35 days)
   - **Phase 3 (Day 81–147):** Test vehicle (4.1 t) introduced (67 days)
   - **Phase 4 (Day 148+):** Test vehicle increased to 9 t (105 days, of which 40 days are within the test period)

3. **FEM Baseline Merge**

   FEM-reconstructed slopes and deflections are loaded for model validation:

   - 61 spatial positions
   - 0.5 m spacing

4. **FEM Load Extraction**

   Net distributed loads (kN/m) are loaded for each span and each observation day. These values are used to validate the loads identified by the PCNN.

### Temperature Definition

Temperature is defined as the mean of the front and back thermocouples at x = 11 m:

$$
T=\frac{T_{\text{front}}+T_{\text{back}}}{2}.
$$

In [8]:
#same as SPTF
def load_data():
    raw = pd.read_csv(data_wd)
    raw['Timestamp'] = pd.to_datetime(raw['Timestamp'])
    raw['T'] = (raw['temp_at_11m_front'] + raw['temp_at_11m_back']) / 2
    raw['day'] = (raw['Timestamp'].dt.date - REF_DATE).apply(lambda x: x.days)
    daily = raw.groupby('day').agg({'tiltx_11m': 'mean', 'tiltx_19m': 'mean', 'T': 'mean'}).reset_index()
    daily = daily[daily['day'] >= 21].copy().reset_index(drop=True)
    daily['phase'] = 1; daily.loc[daily['day'] >= 46, 'phase'] = 2
    daily.loc[daily['day'] >= 81, 'phase'] = 3; daily.loc[daily['day'] >= 148, 'phase'] = 4
    fea_sl = pd.read_csv(FEA_SLOPE_CSV)
    slope_cols = [c for c in fea_sl.columns if c.startswith('slope_')]
    daily = daily.merge(fea_sl[['day', 'net_udl_AB', 'net_udl_BC'] + slope_cols], on='day', how='left')
    fea_df = pd.read_csv(FEA_DEFL_CSV)
    defl_cols = [c for c in fea_df.columns if c.startswith('defl_')]
    daily = daily.merge(fea_df[['day'] + defl_cols], on='day', how='left')
    for ph in [1, 2, 3, 4]: print(f"  Phase {ph}: {(daily['phase'] == ph).sum()} days")
    return daily, slope_cols, defl_cols

daily, slope_cols, defl_cols = load_data()
N = len(daily); days = daily['day'].values; phases = daily['phase'].values
m11 = daily['tiltx_11m'].values; m19 = daily['tiltx_19m'].values; T = daily['T'].values
gpos = np.array([float(c.replace('slope_', '').replace('m', '')) for c in slope_cols])
fe_sl = daily[slope_cols].values; fe_df = daily[defl_cols].values
fe_q_AB = daily['net_udl_AB'].values; fe_q_BC = daily['net_udl_BC'].values
print(f"\n  {N} observations, {len(gpos)} spatial positions")

  Phase 1: 25 days
  Phase 2: 35 days
  Phase 3: 67 days
  Phase 4: 126 days

  253 observations, 61 spatial positions


## 5. Neural Network (BeamNet)

Same 4-hidden-layer, 64-neuron tanh architecture as the SPTF model.

**Input (25 features):** x(1) + Fourier_t(10) + Fourier_T(10) + phase_oh(4)

**Initialisation:** Xavier normal with gain = 0.3. The small gain ensures the NN output starts near zero, so w = phi * NN ~ 0 at epoch 0 — a physically reasonable initial state (zero deflection).

**Why tanh?** The beam PDE requires computing the 4th derivative w'''' through the network. Tanh is C-infinity (infinitely differentiable), so all derivatives exist and are smooth. ReLU would give zero 2nd derivatives, making the PDE loss meaningless.

In [9]:
#Same as SPTF
class BeamNet(nn.Module):
    def __init__(self, nh=4, nn_=64):
        super().__init__()
        n_in = 1 + 2 * N_FOURIER + 2 * N_FOURIER + 4  # 25
        ls = [nn.Linear(n_in, nn_), nn.Tanh()]
        for _ in range(nh - 1):
            ls += [nn.Linear(nn_, nn_), nn.Tanh()]
        ls.append(nn.Linear(nn_, 1))
        self.net = nn.Sequential(*ls)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight, gain=0.3)
                nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.net(x)

print(f"  BeamNet: {sum(p.numel() for p in BeamNet().parameters()):,} parameters")

  BeamNet: 14,209 parameters


## 6. CEI-PCNN Solver Class

The key structural difference from SPTF is the `_eval_w` method, which selects the appropriate trial function based on the span being evaluated:

```python
phi_fn = phi_AB if span == 'AB' else phi_BC
return phi_fn(x) * net(inp)
```

This allows each span to use a trial function that matches its boundary conditions.

The `_derivs` helper computes:

- \(w\)
- \(w'\)
- \(w''\)
- \(w'''\)
- \(w''''\)

using automatic differentiation (autograd), simplifying the training loop implementation.

### Continuity at 2 m Offset

Continuity is evaluated at:

- x = 13 m (Span AB side)
- x = 17 m (Span BC side)

rather than at x = 14.7 m and x = 15.3 m as used in SPTF.

The 2 m offset is necessary because both trial functions are close to zero near x = 15 m, making higher-order derivatives numerically unstable at small offsets. At the selected locations, both trial functions have sufficiently large values and well-conditioned derivatives, resulting in more reliable continuity evaluation.

In [15]:
class CEI_PCNN:
    #Same as SPTF
    def __init__(self, N, phases, T_arr, nc_AB=20, nc_BC=20):
        self.N = N
        self.xi_AB = torch.linspace(0.5, 14.5, nc_AB)
        self.xi_BC = torch.linspace(15.5, 29.5, nc_BC)
        self.x_s1 = 11.0; self.x_s2 = 19.0
        self.t_norm = torch.arange(N, dtype=torch.float64) / max(N - 1, 1)
        T = np.array(T_arr, dtype=np.float64)
        self.T_mean = T.mean(); self.T_std = T.std() + 1e-10
        self.T_norm = torch.tensor((T - self.T_mean) / self.T_std)
        pa = np.array(phases); ph_oh = np.zeros((N, 4))
        for i in range(N): ph_oh[i, pa[i] - 1] = 1.0
        self.phase_oh = torch.tensor(ph_oh)
        weights = np.zeros(N)
        for ph in np.unique(pa): mask = pa == ph; weights[mask] = 1.0 / mask.sum()
        weights /= weights.sum()
        self.sample_weights = torch.tensor(weights)
        self._net = None
        
    #Same as the _inp in SPTF
    def _build_input(self, x_phys, idx):
        t_vals = self.t_norm[idx].view(-1, 1)
        T_vals = self.T_norm[idx].view(-1, 1)
        ft = fourier_encode(t_vals, FOURIER_FREQS)
        fT = fourier_encode(T_vals, FOURIER_FREQS)
        ph = self.phase_oh[idx]
        return torch.cat([x_phys, ft, fT, ph], dim=1)

    #function to evaluate deflection
    def _eval_w(self, x_tensor, idx, net, span):
        inp = self._build_input(x_tensor, idx)         #obtain the 25 features
        inp_d = torch.cat([x_tensor, inp[:, 1:]], dim=1)
        phi_fn = phi_AB if span == 'AB' else phi_BC       #use an if/else boolean function to separate the spans
        return phi_fn(x_tensor) * net(inp_d)              #determine the deflection for each span on

    #obtain uptill the 4th derivate of w for each span
    def _derivs(self, x_tensor, idx, net, span, up_to=4):
        w = self._eval_w(x_tensor, idx, net, span)
        o = torch.ones_like(w)
        d1 = torch.autograd.grad(w, x_tensor, o, create_graph=True)[0]
        if up_to < 2: return w, d1
        d2 = torch.autograd.grad(d1, x_tensor, o, create_graph=True)[0]
        if up_to < 3: return w, d1, d2
        d3 = torch.autograd.grad(d2, x_tensor, o, create_graph=True)[0]
        if up_to < 4: return w, d1, d2, d3
        d4 = torch.autograd.grad(d3, x_tensor, o, create_graph=True)[0]
        return w, d1, d2, d3, d4

    def solve(self, tiltx_11m, tiltx_19m, n_adam=5000, n_lbfgs=80, bs=32, nc_per=10):
        net = BeamNet(4, 64)
        q_AB = nn.Parameter(torch.full((self.N,), 5.0, dtype=torch.float64))
        q_BC = nn.Parameter(torch.full((self.N,), 5.0, dtype=torch.float64))
        s11_t = torch.tensor(tiltx_11m, dtype=torch.float64)
        s19_t = torch.tensor(tiltx_19m, dtype=torch.float64)
        nc_AB = len(self.xi_AB); nc_BC = len(self.xi_BC)
    
        opt = torch.optim.Adam([
            {'params': net.parameters(), 'lr': 1e-3},
            {'params': [q_AB, q_BC], 'lr': 3e-1}])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_adam, eta_min=1e-6)
        history = {'data': [], 'pde': [], 'continuity': [], 'total': []}
    
        for ep in range(n_adam):
            idx = torch.multinomial(self.sample_weights, bs, replacement=True)
            opt.zero_grad()
    
            # DATA: sensor at x=11m (span AB) and x=19m (span BC)
            x1 = torch.full((bs, 1), self.x_s1, dtype=torch.float64).requires_grad_(True)
            w1, dw1 = self._derivs(x1, idx, net, 'AB', up_to=1)
            Ld1 = torch.mean((-dw1.squeeze() * TILTX_SCALE - s11_t[idx])**2)
    
            x2 = torch.full((bs, 1), self.x_s2, dtype=torch.float64).requires_grad_(True)
            w2, dw2 = self._derivs(x2, idx, net, 'BC', up_to=1)
            Ld2 = torch.mean((-dw2.squeeze() * TILTX_SCALE - s19_t[idx])**2)
            Ld = Ld1 + Ld2
    
            # PDE: w'''' + q/EI = 0
            idx_c = idx.repeat_interleave(nc_per)
            ci_a = torch.randint(0, nc_AB, (bs * nc_per,))
            xc_a = self.xi_AB[ci_a].view(-1, 1).requires_grad_(True)
            _, _, _, _, d4a = self._derivs(xc_a, idx_c, net, 'AB', 4)
            qa_c = q_AB[idx].repeat_interleave(nc_per).view(-1, 1)
            Lp_AB = torch.mean((d4a + qa_c * EI_INV).squeeze()**2)
    
            ci_b = torch.randint(0, nc_BC, (bs * nc_per,))
            xc_b = self.xi_BC[ci_b].view(-1, 1).requires_grad_(True)
            _, _, _, _, d4b = self._derivs(xc_b, idx_c, net, 'BC', 4)
            qb_c = q_BC[idx].repeat_interleave(nc_per).view(-1, 1)
            Lp_BC = torch.mean((d4b + qb_c * EI_INV).squeeze()**2)
            Lp = Lp_AB + Lp_BC
    
            # CONTINUITY at x=13 (AB side) and x=17 (BC side) — 2m offset
            xL = torch.full((bs, 1), 14.7, dtype=torch.float64).requires_grad_(True)
            _, _, d2L, d3L = self._derivs(xL, idx, net, 'AB', 3)
            xR = torch.full((bs, 1), 15.3, dtype=torch.float64).requires_grad_(True)
            _, _, d2R, d3R = self._derivs(xR, idx, net, 'BC', 3)
            Lc = torch.mean((d2L.squeeze() - d2R.squeeze())**2) + \
                 torch.mean((d3L.squeeze() - d3R.squeeze())**2)
    
            # TOTAL with ramped weights
            lam_pde = min(0.1 + ep * 1.0 / n_adam, 1.0)
            lam_cont = min(1.0 + ep * 20.0 / n_adam, 20.0)
            loss = 20.0 * Ld + lam_pde * Lp + lam_cont * Lc
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            opt.step(); sched.step()
    
            history['data'].append(Ld.item())
            history['pde'].append(Lp.item())
            history['continuity'].append(Lc.item())
            history['total'].append(loss.item())
            if ep % 1000 == 0:
                print(f"    Adam {ep:5d}: Ld={Ld.item():.4e} Lp={Lp.item():.4e} Lc={Lc.item():.4e}")
    
        # L-BFGS refinement
        print(f"    L-BFGS ({n_lbfgs} steps)...")
        all_p = list(net.parameters()) + [q_AB, q_BC]
        lbfgs = torch.optim.LBFGS(all_p, lr=0.3, max_iter=20, line_search_fn='strong_wolfe')
        for step in range(n_lbfgs):
            def closure():
                lbfgs.zero_grad()
                bs2 = min(64, self.N)
                idx2 = torch.multinomial(self.sample_weights, bs2, replacement=True)
                idx_c2 = idx2.repeat_interleave(8)
                x1 = torch.full((bs2, 1), self.x_s1, dtype=torch.float64).requires_grad_(True)
                _, dw1 = self._derivs(x1, idx2, net, 'AB', 1)
                x2 = torch.full((bs2, 1), self.x_s2, dtype=torch.float64).requires_grad_(True)
                _, dw2 = self._derivs(x2, idx2, net, 'BC', 1)
                Ld = torch.mean((-dw1.squeeze()*TILTX_SCALE - s11_t[idx2])**2) + \
                     torch.mean((-dw2.squeeze()*TILTX_SCALE - s19_t[idx2])**2)
                ci2a = torch.randint(0, nc_AB, (bs2*8,))
                xca = self.xi_AB[ci2a].view(-1, 1).requires_grad_(True)
                _, _, _, _, d4a = self._derivs(xca, idx_c2, net, 'AB', 4)
                Lpa = torch.mean((d4a + q_AB[idx2].repeat_interleave(8).view(-1, 1)*EI_INV).squeeze()**2)
                ci2b = torch.randint(0, nc_BC, (bs2*8,))
                xcb = self.xi_BC[ci2b].view(-1, 1).requires_grad_(True)
                _, _, _, _, d4b = self._derivs(xcb, idx_c2, net, 'BC', 4)
                Lpb = torch.mean((d4b + q_BC[idx2].repeat_interleave(8).view(-1, 1)*EI_INV).squeeze()**2)
                xL = torch.full((bs2, 1), 14.7, dtype=torch.float64).requires_grad_(True)
                _, _, d2L, d3L = self._derivs(xL, idx2, net, 'AB', 3)
                xR = torch.full((bs2, 1), 15.3, dtype=torch.float64).requires_grad_(True)
                _, _, d2R, d3R = self._derivs(xR, idx2, net, 'BC', 3)
                Lc = torch.mean((d2L.squeeze()-d2R.squeeze())**2) + \
                     torch.mean((d3L.squeeze()-d3R.squeeze())**2)
                loss = 20*Ld + Lpa + Lpb + 20*Lc
                loss.backward(); return loss
            try: lbfgs.step(closure)
            except: break
            if step % 20 == 0: print(f"    L-BFGS step {step}")
    
        self._net = net; self._q_AB = q_AB; self._q_BC = q_BC
        return history
    
    def predict_at(self, x_global_positions):
        net = self._net; N = self.N; P = len(x_global_positions)
        slopes = np.zeros((N, P)); defls = np.zeros((N, P))
        idx_all = torch.arange(N)
        for p, xg in enumerate(x_global_positions):
            x_t = torch.full((N, 1), float(xg), dtype=torch.float64).requires_grad_(True)
            span = 'AB' if xg <= 15.0 else 'BC'
            w, dw = self._derivs(x_t, idx_all, net, span, up_to=1)
            slopes[:, p] = (-dw.squeeze() * TILTX_SCALE).detach().numpy()
            defls[:, p] = (-w.squeeze() * 1000).detach().numpy()
        return slopes, defls

## 7. Training Loop

A three-component loss function is used during training:

1. **Data Loss**

   Tilt-X matching at:

   - x = 11 m (using the Span AB trial function)
   - x = 19 m (using the Span BC trial function)

2. **PDE Loss**

   Enforces the beam equation

   $$
   w'''' + \frac{q}{EI} = 0
   $$

   at collocation points in both spans.

3. **Continuity Loss**

   Enforces continuity of moment and shear across the interior support:

   $$
   w''(13) \approx w''(17)
   $$

   $$
   w'''(13) \approx w'''(17)
   $$

   using evaluation points located 2 m from the support on either side.

### Optimization Schedule

The same optimization strategy as SPTF is used:

- Adam optimizer (5000 epochs) with ramped loss weights
- L-BFGS refinement (80 steps)

No moment loss term is required because the boundary condition

$$
w''(30)=0
$$

is hard-coded into the Span BC trial function.

In [11]:
def solve(self, tiltx_11m, tiltx_19m, n_adam=5000, n_lbfgs=80, bs=32, nc_per=10):
    net = BeamNet(4, 64)
    q_AB = nn.Parameter(torch.full((self.N,), 5.0, dtype=torch.float64))
    q_BC = nn.Parameter(torch.full((self.N,), 5.0, dtype=torch.float64))
    s11_t = torch.tensor(tiltx_11m, dtype=torch.float64)
    s19_t = torch.tensor(tiltx_19m, dtype=torch.float64)
    nc_AB = len(self.xi_AB); nc_BC = len(self.xi_BC)

    opt = torch.optim.Adam([
        {'params': net.parameters(), 'lr': 1e-3},
        {'params': [q_AB, q_BC], 'lr': 3e-1}])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_adam, eta_min=1e-6)
    history = {'data': [], 'pde': [], 'continuity': [], 'total': []}

    for ep in range(n_adam):
        idx = torch.multinomial(self.sample_weights, bs, replacement=True)
        opt.zero_grad()

        # DATA: sensor at x=11m (span AB) and x=19m (span BC)
        x1 = torch.full((bs, 1), self.x_s1, dtype=torch.float64).requires_grad_(True)
        w1, dw1 = self._derivs(x1, idx, net, 'AB', up_to=1)
        Ld1 = torch.mean((-dw1.squeeze() * TILTX_SCALE - s11_t[idx])**2)

        x2 = torch.full((bs, 1), self.x_s2, dtype=torch.float64).requires_grad_(True)
        w2, dw2 = self._derivs(x2, idx, net, 'BC', up_to=1)
        Ld2 = torch.mean((-dw2.squeeze() * TILTX_SCALE - s19_t[idx])**2)
        Ld = Ld1 + Ld2

        # PDE: w'''' + q/EI = 0
        idx_c = idx.repeat_interleave(nc_per)
        ci_a = torch.randint(0, nc_AB, (bs * nc_per,))
        xc_a = self.xi_AB[ci_a].view(-1, 1).requires_grad_(True)
        _, _, _, _, d4a = self._derivs(xc_a, idx_c, net, 'AB', 4)
        qa_c = q_AB[idx].repeat_interleave(nc_per).view(-1, 1)
        Lp_AB = torch.mean((d4a + qa_c * EI_INV).squeeze()**2)

        ci_b = torch.randint(0, nc_BC, (bs * nc_per,))
        xc_b = self.xi_BC[ci_b].view(-1, 1).requires_grad_(True)
        _, _, _, _, d4b = self._derivs(xc_b, idx_c, net, 'BC', 4)
        qb_c = q_BC[idx].repeat_interleave(nc_per).view(-1, 1)
        Lp_BC = torch.mean((d4b + qb_c * EI_INV).squeeze()**2)
        Lp = Lp_AB + Lp_BC

        # CONTINUITY at x=13 (AB side) and x=17 (BC side) — 2m offset
        xL = torch.full((bs, 1), 14.7, dtype=torch.float64).requires_grad_(True)
        _, _, d2L, d3L = self._derivs(xL, idx, net, 'AB', 3)
        xR = torch.full((bs, 1), 15.3, dtype=torch.float64).requires_grad_(True)
        _, _, d2R, d3R = self._derivs(xR, idx, net, 'BC', 3)
        Lc = torch.mean((d2L.squeeze() - d2R.squeeze())**2) + \
             torch.mean((d3L.squeeze() - d3R.squeeze())**2)

        # TOTAL with ramped weights
        lam_pde = min(0.1 + ep * 1.0 / n_adam, 1.0)
        lam_cont = min(1.0 + ep * 20.0 / n_adam, 20.0)
        loss = 20.0 * Ld + lam_pde * Lp + lam_cont * Lc
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
        opt.step(); sched.step()

        history['data'].append(Ld.item())
        history['pde'].append(Lp.item())
        history['continuity'].append(Lc.item())
        history['total'].append(loss.item())
        if ep % 1000 == 0:
            print(f"    Adam {ep:5d}: Ld={Ld.item():.4e} Lp={Lp.item():.4e} Lc={Lc.item():.4e}")

    # L-BFGS refinement
    print(f"    L-BFGS ({n_lbfgs} steps)...")
    all_p = list(net.parameters()) + [q_AB, q_BC]
    lbfgs = torch.optim.LBFGS(all_p, lr=0.3, max_iter=20, line_search_fn='strong_wolfe')
    for step in range(n_lbfgs):
        def closure():
            lbfgs.zero_grad()
            bs2 = min(64, self.N)
            idx2 = torch.multinomial(self.sample_weights, bs2, replacement=True)
            idx_c2 = idx2.repeat_interleave(8)
            x1 = torch.full((bs2, 1), self.x_s1, dtype=torch.float64).requires_grad_(True)
            _, dw1 = self._derivs(x1, idx2, net, 'AB', 1)
            x2 = torch.full((bs2, 1), self.x_s2, dtype=torch.float64).requires_grad_(True)
            _, dw2 = self._derivs(x2, idx2, net, 'BC', 1)
            Ld = torch.mean((-dw1.squeeze()*TILTX_SCALE - s11_t[idx2])**2) + \
                 torch.mean((-dw2.squeeze()*TILTX_SCALE - s19_t[idx2])**2)
            ci2a = torch.randint(0, nc_AB, (bs2*8,))
            xca = self.xi_AB[ci2a].view(-1, 1).requires_grad_(True)
            _, _, _, _, d4a = self._derivs(xca, idx_c2, net, 'AB', 4)
            Lpa = torch.mean((d4a + q_AB[idx2].repeat_interleave(8).view(-1, 1)*EI_INV).squeeze()**2)
            ci2b = torch.randint(0, nc_BC, (bs2*8,))
            xcb = self.xi_BC[ci2b].view(-1, 1).requires_grad_(True)
            _, _, _, _, d4b = self._derivs(xcb, idx_c2, net, 'BC', 4)
            Lpb = torch.mean((d4b + q_BC[idx2].repeat_interleave(8).view(-1, 1)*EI_INV).squeeze()**2)
            xL = torch.full((bs2, 1), 14.7, dtype=torch.float64).requires_grad_(True)
            _, _, d2L, d3L = self._derivs(xL, idx2, net, 'AB', 3)
            xR = torch.full((bs2, 1), 15.3, dtype=torch.float64).requires_grad_(True)
            _, _, d2R, d3R = self._derivs(xR, idx2, net, 'BC', 3)
            Lc = torch.mean((d2L.squeeze()-d2R.squeeze())**2) + \
                 torch.mean((d3L.squeeze()-d3R.squeeze())**2)
            loss = 20*Ld + Lpa + Lpb + 20*Lc
            loss.backward(); return loss
        try: lbfgs.step(closure)
        except: break
        if step % 20 == 0: print(f"    L-BFGS step {step}")

    self._net = net; self._q_AB = q_AB; self._q_BC = q_BC
    return history

def predict_at(self, x_global_positions):
    net = self._net; N = self.N; P = len(x_global_positions)
    slopes = np.zeros((N, P)); defls = np.zeros((N, P))
    idx_all = torch.arange(N)
    for p, xg in enumerate(x_global_positions):
        x_t = torch.full((N, 1), float(xg), dtype=torch.float64).requires_grad_(True)
        span = 'AB' if xg <= 15.0 else 'BC'
        w, dw = self._derivs(x_t, idx_all, net, span, up_to=1)
        slopes[:, p] = (-dw.squeeze() * TILTX_SCALE).detach().numpy()
        defls[:, p] = (-w.squeeze() * 1000).detach().numpy()
    return slopes, defls

## 8. Accuracy Metrics and Helper Functions

**met():** Computes R2, RMSE, MAE, MAPE, Bias, and Pearson r between actual and predicted arrays. MAPE is skipped when values are near zero to avoid division issues.

**day_to_date_str():** Converts a day index (relative to Feb 1, 2024) to a human-readable date string for plot labels and JSON output.

**_vlines():** Adds vertical dashed lines at phase transition days (21, 46, 81, 148) on time series plots for visual reference.

In [16]:
def met(y, yp):
    err = yp - y; ss_res = np.sum(err**2); ss_tot = np.sum((y - y.mean())**2)
    mask = np.abs(y) > 1e-6
    mape = float(np.mean(np.abs(err[mask] / y[mask])) * 100) if mask.sum() > 0 else float('nan')
    return {'R2': float(1-ss_res/(ss_tot+1e-12)), 'RMSE': float(np.sqrt(np.mean(err**2))),
            'MAE': float(np.mean(np.abs(err))), 'MAPE': mape,
            'r': float(np.corrcoef(y.ravel(), yp.ravel())[0, 1]) if y.ravel().std() > 1e-10 else 0.0}

def day_to_date_str(d):
    return (pd.Timestamp('2024-02-01') + timedelta(days=int(d))).strftime('%B %d')

def _vlines(ax):
    for d in [21, 46, 81, 148]: ax.axvline(d, color='gray', ls='--', lw=0.8, alpha=0.5)

## 9. Physics Compliance Evaluation

Evaluates three categories of physical consistency after training:

### Boundary Conditions (RMS over all 253 days)
- **Hard BCs** (enforced by phi_AB and phi_BC): w(0), w'(0), w(15), w'(15), w(30) — violations should be ~1e-10 or smaller (machine precision)
- **w''(30)** — algebraically enforced by phi_BC's (22.5-xi) factor, evaluated at x = 29.95 to avoid the exact root

### Continuity at Interior Support (x = 13 / x = 17)
- **Moment jump:** w''(13) - w''(17) — should be zero for a continuous beam
- **Shear jump:** w'''(13) - w'''(17) — should be zero

Evaluated at 2m offset from x = 15 (not 0.3m as in SPTF) because both phi_AB and phi_BC are near zero close to x = 15, making derivative computation numerically unstable at small offsets.

### PDE Residual
For each span and each of the 253 days: evaluate w''''(x) + q(t)/EI at 30 collocation points, compute MSE, then report RMS and Max across all days. Uses 30 points (vs 20 in SPTF) for finer resolution.

In [17]:
def evaluate_physics(solver, N):
    net = solver._net; idx_all = torch.arange(N); results = {}
    bc_v = {}
    for xv, bc, sp in [(0.0, 'w(0)', 'AB'), (L_SPAN, 'w(15)', 'AB'), (L_TOTAL, 'w(30)', 'BC')]:
        x_t = torch.full((N, 1), xv, dtype=torch.float64).requires_grad_(True)
        w = solver._eval_w(x_t, idx_all, net, sp)
        bc_v[bc] = float(torch.sqrt(torch.mean(w**2)).item())
    for xv, bc, sp in [(0.0, "w'(0)", 'AB'), (L_SPAN, "w'(15)", 'AB')]:
        x_t = torch.full((N, 1), xv, dtype=torch.float64).requires_grad_(True)
        w, dw = solver._derivs(x_t, idx_all, net, sp, 1)
        bc_v[bc] = float(torch.sqrt(torch.mean(dw**2)).item())
    x30 = torch.full((N, 1), L_TOTAL - 0.05, dtype=torch.float64).requires_grad_(True)
    _, _, d2_30 = solver._derivs(x30, idx_all, net, 'BC', 2)
    bc_v["w''(30)"] = float(torch.sqrt(torch.mean(d2_30**2)).item())
    results['bc_violations_rms'] = bc_v

    xL = torch.full((N, 1), 13.0, dtype=torch.float64).requires_grad_(True)
    _, _, d2L, d3L = solver._derivs(xL, idx_all, net, 'AB', 3)
    xR = torch.full((N, 1), 17.0, dtype=torch.float64).requires_grad_(True)
    _, _, d2R, d3R = solver._derivs(xR, idx_all, net, 'BC', 3)
    mj = d2L.squeeze().detach().numpy() - d2R.squeeze().detach().numpy()
    sj = d3L.squeeze().detach().numpy() - d3R.squeeze().detach().numpy()
    results['continuity'] = {
        'moment_jump_rms': float(np.sqrt(np.mean(mj**2))),
        'shear_jump_rms': float(np.sqrt(np.mean(sj**2))),
        'moment_jump_max': float(np.max(np.abs(mj))),
        'shear_jump_max': float(np.max(np.abs(sj))),
    }

    pde_res = {}
    for sn, xi_ev, qp in [('AB', torch.linspace(1, 14, 30), solver._q_AB),
                            ('BC', torch.linspace(16, 29, 30), solver._q_BC)]:
        residuals = []
        for i in range(N):
            x_t = xi_ev.view(-1, 1).clone().requires_grad_(True)
            idx_i = torch.full((30,), i, dtype=torch.long)
            _, _, _, _, d4 = solver._derivs(x_t, idx_i, net, sn, 4)
            res = d4.squeeze().detach().numpy() + qp[i].detach().item() * EI_INV
            residuals.append(np.mean(res**2))
        pde_res[sn] = {'rms_residual': float(np.sqrt(np.mean(residuals))),
                        'max_residual': float(np.sqrt(np.max(residuals)))}
    results['pde_residual'] = pde_res
    return results

## 10. Run Training

Instantiate the solver with all 253 observations and run the full training pipeline:
1. **Adam optimiser** (5000 epochs, batch size 32): Stochastic mini-batch training with cosine LR annealing
2. **L-BFGS refinement** (80 steps): Quasi-Newton method for fine convergence with strong Wolfe line search

Expected runtime: ~3-5 minutes on CPU.

In [18]:
t0 = datetime.now()
print("="*70)
print("CEI-PCNN: Piecewise Trial Functions")
print("="*70)
solver = CEI_PCNN(N, phases, T)
history = solver.solve(m11, m19, n_adam=5000, n_lbfgs=80)
print(f"  Training time: {(datetime.now()-t0).total_seconds():.0f}s")

CEI-PCNN: Piecewise Trial Functions
    Adam     0: Ld=6.4762e-01 Lp=6.1082e-09 Lc=4.2906e-07
    Adam  1000: Ld=9.9231e-03 Lp=1.8618e-08 Lc=8.0281e-07
    Adam  2000: Ld=9.6487e-03 Lp=1.7612e-08 Lc=7.4581e-07
    Adam  3000: Ld=5.9402e-03 Lp=2.0247e-08 Lc=7.5629e-07
    Adam  4000: Ld=5.4907e-03 Lp=1.9074e-08 Lc=7.6748e-07
    L-BFGS (80 steps)...
    L-BFGS step 0
    L-BFGS step 20
    L-BFGS step 40
    L-BFGS step 60
  Training time: 1282s


## 11. Evaluate Accuracy

Compare the PCNN output against FEM baseline:
- **Sensor positions** (x = 11, 19 m): How well does the PCNN reproduce the measured tiltx used for training?
- **Validation positions** (x = 5, 10, 20, 25 m): How well does the PCNN reconstruct at uninstrumented locations? These are the key test of spatial generalisation.
- **Load discovery:** How well do the learned q_AB(t), q_BC(t) match the FEM net UDL?

In [19]:
pcnn_slopes, pcnn_defls = solver.predict_at(gpos)
ip11 = np.argmin(np.abs(gpos - 11.0)); ip19 = np.argmin(np.abs(gpos - 19.0))
pred_11 = pcnn_slopes[:, ip11]; pred_19 = pcnn_slopes[:, ip19]
m_s_AB = met(m11, pred_11); m_s_BC = met(m19, pred_19)
pcnn_q_AB = solver._q_AB.detach().numpy(); pcnn_q_BC = solver._q_BC.detach().numpy()
m_q_AB = met(fe_q_AB, pcnn_q_AB); m_q_BC = met(fe_q_BC, pcnn_q_BC)

print(f"  Sensor: AB r={m_s_AB['r']:.4f} | BC r={m_s_BC['r']:.4f}")
val_sl = {}; val_df = {}
for xv in VAL_POS:
    ip = np.argmin(np.abs(gpos - xv))
    ms = met(fe_sl[:, ip], pcnn_slopes[:, ip]); md_ = met(fe_df[:, ip], pcnn_defls[:, ip])
    val_sl[xv] = ms; val_df[xv] = md_
    print(f"  x={xv:.0f}m: sl R2={ms['R2']:.4f} | df R2={md_['R2']:.4f}")

  Sensor: AB r=0.9880 | BC r=0.9566
  x=5m: sl R2=-0.8216 | df R2=-2.7127
  x=10m: sl R2=-1.7397 | df R2=-0.2764
  x=20m: sl R2=0.0974 | df R2=0.2540
  x=25m: sl R2=-6.6789 | df R2=0.0028


## 12. Evaluate Physics Compliance

Check boundary condition satisfaction, continuity at the interior support, and PDE residual quality. Hard BCs should have violations ~1e-10. The continuity metrics are the key comparison point: PPTF typically shows 8x worse moment continuity and 43x worse shear continuity than SPTF, because the piecewise construction introduces inherent derivative discontinuities at x = 15 that the continuity loss can only partially suppress.

In [20]:
phys = evaluate_physics(solver, N)
for k, v in phys['bc_violations_rms'].items():
    print(f"  {k:>10s}: {v:.2e}  [{'HARD' if v<1e-8 else 'SOFT'}]")
c = phys['continuity']
print(f"  Continuity: moment RMS={c['moment_jump_rms']:.4e}, shear RMS={c['shear_jump_rms']:.4e}")
for sn, pde in phys['pde_residual'].items():
    print(f"  PDE {sn}: RMS={pde['rms_residual']:.4e}")

        w(0): 0.00e+00  [HARD]
       w(15): 0.00e+00  [HARD]
       w(30): 0.00e+00  [HARD]
       w'(0): 0.00e+00  [HARD]
      w'(15): 0.00e+00  [HARD]
     w''(30): 1.28e-04  [SOFT]
  Continuity: moment RMS=4.6914e-04, shear RMS=6.9639e-04
  PDE AB: RMS=1.2089e-04
  PDE BC: RMS=3.5383e-05


## 13. Plot: Training Convergence

3-component convergence (no L_moment term since w''(30) = 0 is hardcoded by phi_BC). All loss components on log scale. The data loss should decrease steadily; PDE and continuity losses decrease as their weights ramp up during training.

In [21]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(history['data'], 'b-', lw=0.5, alpha=0.7, label='L_data')
ax.semilogy(history['pde'], 'r-', lw=0.5, alpha=0.7, label='L_pde')
ax.semilogy(history['continuity'], 'g-', lw=0.5, alpha=0.7, label='L_continuity')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('CEI-PCNN Training Convergence', fontsize=13, fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT, '01_convergence.png'), dpi=200); plt.close()
print("  Saved: 01_convergence.png")

findfont: Font family ['cmsy10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmr10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmtt10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmmi10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmb10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmss10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmex10'] not found. Falling back to DejaVu Sans.


  Saved: 01_convergence.png


## 14. Plot: Sensor Fit

Measured vs PCNN-reconstructed tiltx at x = 11 m (span AB, using phi_AB) and x = 19 m (span BC, using phi_BC). Phase transitions marked with vertical dashed lines. The PCNN should closely track the measured data (r > 0.96 typically).

In [22]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('CEI-PCNN: Sensor Fit', fontsize=13, fontweight='bold')
for ax, m, pred, ms, title in [(axes[0], m11, pred_11, m_s_AB, 'AB x=11m'),
                                (axes[1], m19, pred_19, m_s_BC, 'BC x=19m')]:
    ax.plot(days, m, 'b-', lw=1, alpha=0.7, label='Measured')
    ax.plot(days, pred, 'r-', lw=1, alpha=0.7, label=f'PCNN (r={ms["r"]:.4f})')
    _vlines(ax); ax.set_ylabel('tiltx (mm/m)'); ax.set_title(title)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
axes[1].set_xlabel('Day')
plt.tight_layout(); plt.savefig(os.path.join(OUT, '02_sensor_fit.png'), dpi=200); plt.close()
print("  Saved: 02_sensor_fit.png")

  Saved: 02_sensor_fit.png


## 15. Plot: Validation Time Series

PCNN vs FEM at 4 uninstrumented positions (x = 5, 10, 20, 25 m). Left column: slope, Right column: deflection. These positions are NOT used during training — they measure the PCNN's spatial generalisation ability.

Positions in span AB (5, 10 m) should show reasonable agreement since phi_AB has the correct fixed-fixed shape. Positions in span BC (20, 25 m) may show different behaviour depending on whether phi_BC captures the fixed-pinned shape correctly.

In [23]:
fig, axes = plt.subplots(4, 2, figsize=(16, 16))
fig.suptitle('CEI-PCNN vs FEM: Validation', fontsize=13, fontweight='bold')
for row, xv in enumerate(VAL_POS):
    ip = np.argmin(np.abs(gpos - xv))
    ms = val_sl[xv]; md_ = val_df[xv]
    axes[row, 0].plot(days, fe_sl[:, ip], 'b-', lw=1, alpha=0.7, label='FEM')
    axes[row, 0].plot(days, pcnn_slopes[:, ip], 'r-', lw=1, alpha=0.7, label=f'PCNN (R2={ms["R2"]:.3f})')
    _vlines(axes[row, 0]); axes[row, 0].set_ylabel('Slope (mm/m)')
    axes[row, 0].set_title(f'x={xv:.0f}m Slope'); axes[row, 0].legend(fontsize=7); axes[row, 0].grid(True, alpha=0.3)
    axes[row, 1].plot(days, fe_df[:, ip], 'b-', lw=1, alpha=0.7, label='FEM')
    axes[row, 1].plot(days, pcnn_defls[:, ip], 'r-', lw=1, alpha=0.7, label=f'PCNN (R2={md_["R2"]:.3f})')
    _vlines(axes[row, 1]); axes[row, 1].set_ylabel('Defl (mm)')
    axes[row, 1].set_title(f'x={xv:.0f}m Deflection'); axes[row, 1].legend(fontsize=7); axes[row, 1].grid(True, alpha=0.3)
axes[3, 0].set_xlabel('Day'); axes[3, 1].set_xlabel('Day')
plt.tight_layout(); plt.savefig(os.path.join(OUT, '03_validation_ts.png'), dpi=200); plt.close()
print("  Saved: 03_validation_ts.png")

  Saved: 03_validation_ts.png


## 16. Plot: Validation Scatter (Phase-Coloured)

Scatter plots of PCNN vs FEM at each validation position, coloured by loading phase. Points on the diagonal indicate perfect agreement. Phase-dependent clusters reveal whether the PCNN correctly handles the loading changes between phases. Systematic offset from the diagonal indicates a bias in the reconstruction.

In [24]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('CEI-PCNN vs FEM: Scatter', fontsize=13, fontweight='bold')
for col, xv in enumerate(VAL_POS):
    ip = np.argmin(np.abs(gpos - xv))
    for ph in [1, 2, 3, 4]:
        mk = phases == ph
        axes[0, col].scatter(fe_sl[mk, ip], pcnn_slopes[mk, ip], s=8, c=CP[ph], alpha=0.6,
                             label=LP[ph] if col == 0 else '')
        axes[1, col].scatter(fe_df[mk, ip], pcnn_defls[mk, ip], s=8, c=CP[ph], alpha=0.6)
    for r in [0, 1]:
        fe = fe_sl[:, ip] if r == 0 else fe_df[:, ip]
        lm = [fe.min(), fe.max()]; axes[r, col].plot(lm, lm, 'k--', lw=1); axes[r, col].grid(True, alpha=0.3)
    axes[0, col].set_title(f'x={xv:.0f}m Slope', fontsize=9)
    axes[1, col].set_title(f'x={xv:.0f}m Defl', fontsize=9)
axes[0, 0].legend(fontsize=5, ncol=2)
axes[0, 0].set_ylabel('PCNN Slope'); axes[1, 0].set_ylabel('PCNN Defl')
for col in range(4): axes[1, col].set_xlabel('FEM')
plt.tight_layout(); plt.savefig(os.path.join(OUT, '04_validation_scatter.png'), dpi=200); plt.close()
print("  Saved: 04_validation_scatter.png")

  Saved: 04_validation_scatter.png


## 17. Plot: Mean Spatial Profiles

Average slope and deflection profiles (over all 253 days) comparing FEM (black solid) and PCNN (red dashed). This reveals systematic spatial bias from the trial function shape. Blue dashed lines mark the sensor positions (x = 11, 19 m). A discontinuity at x = 15 would indicate the piecewise construction is not achieving smooth continuity.

In [25]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Mean Profiles: FEM vs CEI-PCNN', fontsize=13, fontweight='bold')
for ax, fe, pc, yl in [(axes[0], fe_sl, pcnn_slopes, 'Slope (mm/m)'),
                        (axes[1], fe_df, pcnn_defls, 'Deflection (mm)')]:
    ax.plot(gpos, fe.mean(0), 'k-', lw=2, label='FEM')
    ax.plot(gpos, pc.mean(0), 'r--', lw=2, label='CEI-PCNN')
    ax.axvline(11, color='blue', ls=':', lw=1.5, alpha=0.7)
    ax.axvline(19, color='blue', ls=':', lw=1.5, alpha=0.7)
    ax.axvline(15, color='black', lw=2, alpha=0.3)
    ax.set_ylabel(yl); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
axes[1].set_xlabel('x (m)')
plt.tight_layout(); plt.savefig(os.path.join(OUT, '05_mean_profiles.png'), dpi=200); plt.close()
print("  Saved: 05_mean_profiles.png")

  Saved: 05_mean_profiles.png


## 18. Plot: Load Discovery

PCNN-discovered load q(t) vs FEM net UDL. Top row: time series showing how the discovered load evolves across loading phases. Bottom row: scatter by phase, with the diagonal indicating perfect match. Good load discovery (high r) confirms the PDE loss is working — the PCNN has found physically meaningful loads, not just arbitrary values that minimise the total loss.

In [26]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Discovered Load: PCNN q(t) vs FEM Net UDL', fontsize=13, fontweight='bold')
for col, span, fq, pq, mq in [(0, 'AB', fe_q_AB, pcnn_q_AB, m_q_AB),
                                (1, 'BC', fe_q_BC, pcnn_q_BC, m_q_BC)]:
    axes[0, col].plot(days, fq, 'b-', lw=1, alpha=0.7, label='FEM')
    axes[0, col].plot(days, pq, 'r-', lw=1, alpha=0.7, label=f'PCNN (r={mq["r"]:.4f})')
    for d in [46, 81, 148]: axes[0, col].axvline(d, color='gray', ls='--', lw=0.5)
    axes[0, col].set_ylabel('Net UDL (kN/m)'); axes[0, col].set_title(f'Span {span}')
    axes[0, col].legend(fontsize=7); axes[0, col].grid(True, alpha=0.3)
    for ph in [1, 2, 3, 4]:
        mk = phases == ph
        axes[1, col].scatter(fq[mk], pq[mk], s=12, c=CP[ph], alpha=0.6, label=LP[ph])
    lm = [min(fq.min(), pq.min()), max(fq.max(), pq.max())]
    axes[1, col].plot(lm, lm, 'k--', lw=1.5)
    axes[1, col].set_xlabel('FEM (kN/m)'); axes[1, col].set_ylabel('PCNN (kN/m)')
    axes[1, col].set_title(f'{span}: RMSE={mq["RMSE"]:.3f}'); axes[1, col].legend(fontsize=6)
    axes[1, col].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT, '06_load_comparison.png'), dpi=200); plt.close()
print("  Saved: 06_load_comparison.png")

  Saved: 06_load_comparison.png


## 19. Plot: Daily Profiles

Selected daily profiles from different phases and seasons, showing FEM (solid) vs PCNN (dashed). This reveals whether the PCNN captures day-to-day variation (driven by temperature changes), not just the mean shape. Profiles from Phase 1 (dead load only) should differ from Phase 4 (vehicle 9 t) due to the additional concentrated loads.

In [27]:
target_days = [21, 40, 55, 75, 100, 130, 160, 190, 220, 250]
sel = sorted(set([np.argmin(np.abs(days - td)) for td in target_days]))
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Daily Profiles: FEM (solid) vs CEI-PCNN (dashed)', fontsize=13, fontweight='bold')
for j, di in enumerate(sel[:6]):
    ds = day_to_date_str(days[di]); cc = DISTINCT_COLORS[j % len(DISTINCT_COLORS)]
    axes[0].plot(gpos, fe_sl[di], '-', color=cc, lw=2, label=f'{ds} (Ph{phases[di]})')
    axes[0].plot(gpos, pcnn_slopes[di], '--', color=cc, lw=2)
    axes[1].plot(gpos, fe_df[di], '-', color=cc, lw=2, label=f'{ds} (Ph{phases[di]})')
    axes[1].plot(gpos, pcnn_defls[di], '--', color=cc, lw=2)
axes[0].plot([], [], '-k', lw=2, label='FEM (solid)')
axes[0].plot([], [], '--k', lw=2, label='PCNN (dashed)')
for ax in axes:
    ax.axvline(11, color='blue', ls=':', lw=1, alpha=0.5)
    ax.axvline(19, color='red', ls=':', lw=1, alpha=0.5)
    ax.axvline(15, color='black', lw=2, alpha=0.3)
    ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Slope (mm/m)'); axes[1].set_ylabel('Deflection (mm)'); axes[1].set_xlabel('x (m)')
plt.tight_layout(); plt.savefig(os.path.join(OUT, '12_daily_profiles.png'), dpi=200); plt.close()
print("  Saved: 12_daily_profiles.png")

  Saved: 12_daily_profiles.png


## 20. Save Results

Two output files:
1. **pcnn_results.csv:** Full results table — measured tiltx, PCNN predictions at all 61 spatial positions, discovered loads q_AB(t) and q_BC(t), and FEM loads for comparison (253 rows x ~130 columns)
2. **pcnn_summary.json:** Structured metrics — accuracy (sensor, validation, load), physics compliance (BC violations, continuity, PDE residual), trial function specification, and model metadata for automated comparison with SPTF/PETF/AR-PCNN

In [28]:
mf_sl = met(fe_sl, pcnn_slopes); mf_df = met(fe_df, pcnn_defls)

df_out = pd.DataFrame({
    'day': days, 'phase': phases, 'tiltx_11m': m11, 'tiltx_19m': m19,
    'pcnn_pred_11m': pred_11, 'pcnn_pred_19m': pred_19,
    'q_AB': pcnn_q_AB, 'q_BC': pcnn_q_BC,
    'fe_q_AB': fe_q_AB, 'fe_q_BC': fe_q_BC,
})
for ip, xv in enumerate(gpos):
    df_out[f'pcnn_slope_{xv:.1f}m'] = pcnn_slopes[:, ip]
    df_out[f'pcnn_defl_{xv:.1f}m'] = pcnn_defls[:, ip]
df_out.to_csv(os.path.join(OUT, 'pcnn_results.csv'), index=False)

with open(os.path.join(OUT, 'pcnn_summary.json'), 'w') as f:
    json.dump({
        'method': 'CEI-PCNN (piecewise phi, Fourier t/T, continuity at 2m offset)',
        'trial_functions': {
            'AB': 'x^2*(x-15)^2 [fixed-fixed]',
            'BC': 'xi^2*(15-xi)*(22.5-xi) [fixed-pinned]',
        },
        'EI': float(EI_NOM),
        'accuracy': {
            'sensor': {'AB': m_s_AB, 'BC': m_s_BC},
            'validation': {f'x{xv:.0f}m': {'slope': val_sl[xv], 'defl': val_df[xv]} for xv in VAL_POS},
            'load': {'AB': m_q_AB, 'BC': m_q_BC},
        },
        'physics': phys,
    }, f, indent=2, default=str)

print(f"  Saved to {OUT}/:")
print(f"    pcnn_results.csv ({len(df_out)} rows x {len(df_out.columns)} cols)")
print(f"    pcnn_summary.json")
print(f"  Runtime: {(datetime.now()-t0).total_seconds():.0f}s")

  Saved to C:\Users\ridoc\OneDrive\Desktop\Folders\SMACCs\Thesis\Physics Guided Framework\Draft\Models\outputs\processing\Tiltx\Piecewise_Trial_function_PCNN/:
    pcnn_results.csv (253 rows x 132 cols)
    pcnn_summary.json
  Runtime: 4127s


C:\Users\ridoc\AppData\Local\Temp\ipykernel_47340\2307529863.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[f'pcnn_slope_{xv:.1f}m'] = pcnn_slopes[:, ip]
C:\Users\ridoc\AppData\Local\Temp\ipykernel_47340\2307529863.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[f'pcnn_defl_{xv:.1f}m'] = pcnn_defls[:, ip]
C:\Users\ridoc\AppData\Local\Temp\ipykernel_47340\2307529863.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor p